# SQL vs pandas : le guide pour comparer les deux

Notebook de l'article [SQL vs pandas : le guide pour comparer les deux](https://moncoachdata.com/blog/sql-vs-pandas/) (MonCoachData).

Chaque requête SQL (exécutée avec SQLite) est comparée à son équivalent pandas sur le jeu de données Titanic. Exécute les cellules **dans l'ordre**. Code testé en octobre 2026 avec Python 3.13 et pandas 3.

In [48]:
import sys, sqlite3, pandas as pd

print("Python", sys.version.split()[0])
print("pandas", pd.__version__)
print("SQLite", sqlite3.sqlite_version)

Python 3.13.16
pandas 2.2.3
SQLite 3.45.1


## Charger le jeu de données Titanic dans pandas et dans SQL

In [49]:
url = "https://raw.githubusercontent.com/moncoachdata/blog-data-science/main/sql-vs-pandas/titanic.csv"
titanic = pd.read_csv(url)

pd.set_option("display.max_rows", 8)
titanic.shape

(891, 15)

In [50]:
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [51]:
conn = sqlite3.connect(":memory:")
titanic.to_sql("titanic", conn, index=False)

def sql(requete):
    return pd.read_sql_query(requete, conn)

## Sélectionner des colonnes

In [52]:
sql("""
SELECT pclass, fare, age, survived
FROM titanic
""")

,pclass,fare,age,survived
0,3,7.2500,22.0,0
1,1,71.2833,38.0,1
2,3,7.9250,26.0,1
3,1,53.1000,35.0,1
...,...,...,...,...
887,1,30.0000,19.0,1
888,3,23.4500,NaN,0
889,1,30.0000,26.0,1
890,3,7.7500,32.0,0


In [53]:
col_list = ['pclass', 'fare', 'age', 'survived']
titanic[col_list]

,pclass,fare,age,survived
0,3,7.2500,22.0,0
1,1,71.2833,38.0,1
2,3,7.9250,26.0,1
3,1,53.1000,35.0,1
...,...,...,...,...
887,1,30.0000,19.0,1
888,3,23.4500,NaN,0
889,1,30.0000,26.0,1
890,3,7.7500,32.0,0


In [54]:
titanic[['pclass', 'fare', 'age', 'survived']].equals(titanic[col_list])

True

In [55]:
titanic.loc[:, ['pclass', 'fare', 'age', 'survived']].equals(titanic[col_list])

True

In [56]:
titanic.loc[50:100, col_list]

,pclass,fare,age,survived
50,3,39.6875,7.0,0
51,3,7.8000,21.0,0
52,1,76.7292,49.0,1
53,2,26.0000,29.0,1
...,...,...,...,...
97,1,63.3583,23.0,1
98,2,23.0000,34.0,1
99,2,26.0000,34.0,0
100,3,7.8958,28.0,0


## Filtrer des lignes (WHERE)

In [57]:
sql("""
SELECT pclass, fare, age, survived
FROM titanic
WHERE sex = 'male' AND age >= 18
""")

,pclass,fare,age,survived
0,3,7.2500,22.0,0
1,3,8.0500,35.0,0
2,1,51.8625,54.0,0
3,3,8.0500,20.0,0
...,...,...,...,...
391,3,7.0500,25.0,0
392,2,13.0000,27.0,0
393,1,30.0000,26.0,1
394,3,7.7500,32.0,0


In [58]:
male_mask = titanic.sex == 'male'
age_mask = titanic.age >= 18
col_list = ['pclass', 'fare', 'age', 'survived']

titanic.loc[male_mask & age_mask, col_list]

,pclass,fare,age,survived
0,3,7.2500,22.0,0
4,3,8.0500,35.0,0
6,1,51.8625,54.0,0
12,3,8.0500,20.0,0
...,...,...,...,...
884,3,7.0500,25.0,0
886,2,13.0000,27.0,0
889,1,30.0000,26.0,1
890,3,7.7500,32.0,0


In [59]:
titanic.loc[~male_mask, 'sex'].value_counts()

,count
sex,
female,314


In [60]:
titanic.query("sex == 'male' and age >= 18")[col_list]

,pclass,fare,age,survived
0,3,7.2500,22.0,0
4,3,8.0500,35.0,0
6,1,51.8625,54.0,0
12,3,8.0500,20.0,0
...,...,...,...,...
884,3,7.0500,25.0,0
886,2,13.0000,27.0,0
889,1,30.0000,26.0,1
890,3,7.7500,32.0,0


In [61]:
print("Passagers dont l'âge est inconnu :", titanic.age.isna().sum())
print("Lignes avec age >= 18 :", (titanic.age >= 18).sum())
print("Lignes avec age < 18  :", (titanic.age < 18).sum())

Passagers dont l'âge est inconnu : 177
Lignes avec age >= 18 : 601
Lignes avec age < 18  : 113


## Trier (ORDER BY)

Avec pandas 3, modifier une tranche ne modifie jamais le DataFrame d'origine (copie sur écriture).

In [62]:
sql("""
SELECT pclass, fare, age, survived
FROM titanic
WHERE sex = 'male' AND age >= 18
ORDER BY fare DESC, age
""")

,pclass,fare,age,survived
0,1,512.3292,35.0,1
1,1,512.3292,36.0,1
2,1,263.0000,19.0,0
3,1,263.0000,64.0,0
...,...,...,...,...
391,1,0.0000,38.0,0
392,1,0.0000,39.0,0
393,1,0.0000,40.0,0
394,3,0.0000,49.0,0


In [63]:
df_slice = titanic.loc[male_mask & age_mask, col_list]
df_slice.sort_values(['fare', 'age'], ascending=[False, True])

,pclass,fare,age,survived
737,1,512.3292,35.0,1
679,1,512.3292,36.0,1
27,1,263.0000,19.0,0
438,1,263.0000,64.0,0
...,...,...,...,...
822,1,0.0000,38.0,0
806,1,0.0000,39.0,0
263,1,0.0000,40.0,0
597,3,0.0000,49.0,0


In [64]:
df_slice = titanic.loc[male_mask & age_mask, col_list]
df_slice.loc[:, 'fare'] = 0

print("Tarif max dans la tranche :", df_slice.fare.max())
print("Tarif max dans titanic    :", titanic.fare.max())

Tarif max dans la tranche : 0.0
Tarif max dans titanic    : 512.3292


## Limiter (LIMIT)

In [65]:
sql("""
SELECT pclass, fare, age, survived
FROM titanic
WHERE sex = 'male' AND age >= 18
ORDER BY fare DESC, age
LIMIT 10
""")

,pclass,fare,age,survived
0,1,512.3292,35.0,1
1,1,512.3292,36.0,1
2,1,263.0000,19.0,0
3,1,263.0000,64.0,0
...,...,...,...,...
6,1,153.4625,38.0,0
7,1,135.6333,22.0,0
8,1,133.6500,50.0,1
9,1,120.0000,36.0,1


In [66]:
df_slice = titanic.loc[male_mask & age_mask, col_list]
df_slice.sort_values(['fare', 'age'], ascending=[False, True]).head(10)

,pclass,fare,age,survived
737,1,512.3292,35.0,1
679,1,512.3292,36.0,1
27,1,263.0000,19.0,0
438,1,263.0000,64.0,0
...,...,...,...,...
332,1,153.4625,38.0,0
373,1,135.6333,22.0,0
660,1,133.6500,50.0,1
390,1,120.0000,36.0,1


In [67]:
df_slice.nsmallest(10, ['fare', 'age'])

,pclass,fare,age,survived
302,3,0.0000,19.0,0
271,3,0.0000,25.0,1
179,3,0.0000,36.0,0
822,1,0.0000,38.0,0
...,...,...,...,...
597,3,0.0000,49.0,0
378,3,4.0125,20.0,0
872,1,5.0000,33.0,0
326,3,6.2375,61.0,0


## Fonctions d'agrégation

In [68]:
sql("""
SELECT MAX(fare), MIN(fare), AVG(age), SUM(survived)
FROM titanic
""")

,MAX(fare),MIN(fare),AVG(age),SUM(survived)
0,512.3292,0.0,29.699118,342


In [69]:
print(titanic.fare.agg(['max', 'min']))
print("Âge moyen :", titanic.age.mean())
print("Survivants :", titanic.survived.sum())

max    512.3292
min      0.0000
Name: fare, dtype: float64
Âge moyen : 29.69911764705882
Survivants : 342


In [70]:
print(titanic.age.mean().round(1))

29.7


In [71]:
print("Médiane de l'âge :", titanic.age.median())
print("Écart type de l'âge :", round(titanic.age.std(), 2))
print("Variance de l'âge :", round(titanic.age.var(), 2))

Médiane de l'âge : 28.0
Écart type de l'âge : 14.53
Variance de l'âge : 211.02


In [72]:
titanic.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [73]:
titanic.describe(include=["object", "string"])

,sex,embarked,class,who,deck,embark_town,alive
count,891,889,891,891,203,889,891
unique,2,3,3,3,7,3,2
top,male,S,Third,man,C,Southampton,no
freq,577,644,491,537,59,644,549


## GROUP BY et groupby()

In [74]:
sql("""
SELECT sex, pclass, COUNT(*), SUM(survived), AVG(age)
FROM titanic
GROUP BY sex, pclass
""")

,sex,pclass,COUNT(*),SUM(survived),AVG(age)
0,female,1,94,91,34.611765
1,female,2,76,70,28.722973
2,female,3,144,72,21.750000
3,male,1,122,45,41.281386
4,male,2,108,17,30.740707
5,male,3,347,47,26.507589


In [75]:
print(titanic.groupby(['sex', 'pclass']).survived.agg(['count', 'sum']))
print(titanic.groupby(['sex', 'pclass']).age.mean())

               count  sum
sex    pclass            
female 1          94   91
       2          76   70
       3         144   72
male   1         122   45
       2         108   17
       3         347   47
sex     pclass
female  1         34.611765
        2         28.722973
        3         21.750000
male    1         41.281386
        2         30.740707
        3         26.507589
Name: age, dtype: float64


In [76]:
new_df = titanic.groupby(['sex', 'pclass'])[['survived', 'age']]
new_df.agg(['count', 'sum', 'mean'])

survived                 age                    
                 count sum      mean count      sum       mean
sex    pclass                                                 
female 1            94  91  0.968085    85  2942.00  34.611765
       2            76  70  0.921053    74  2125.50  28.722973
       3           144  72  0.500000   102  2218.50  21.750000
male   1           122  45  0.368852   101  4169.42  41.281386
       2           108  17  0.157407    99  3043.33  30.740707
       3           347  47  0.135447   253  6706.42  26.507589

In [77]:
titanic.groupby(['sex', 'pclass']).agg(
    passagers=('survived', 'size'),
    survivants=('survived', 'sum'),
    age_moyen=('age', 'mean'),
)

passagers  survivants  age_moyen
sex    pclass                                  
female 1              94          91  34.611765
       2              76          70  28.722973
       3             144          72  21.750000
male   1             122          45  41.281386
       2             108          17  30.740707
       3             347          47  26.507589

## Valeurs manquantes dans un regroupement

SQL garde un groupe `NULL`, pandas l'écarte par défaut (`dropna=False` pour l'inclure).

In [78]:
sql("""
SELECT embarked, COUNT(*)
FROM titanic
GROUP BY embarked
""")

,embarked,COUNT(*)
0,None,2
1,C,168
2,Q,77
3,S,644


In [79]:
titanic.groupby('embarked').size()

,0
embarked,
C,168
Q,77
S,644


In [80]:
titanic.groupby('embarked', dropna=False).size()

,0
embarked,
C,168
Q,77
S,644
NaN,2


## Vérifier que SQL et pandas donnent le même résultat

In [81]:
import numpy as np

def identique(a, b):
    a = a.reset_index(drop=True).copy()
    b = b.reset_index(drop=True).copy()
    b.columns = a.columns
    return np.allclose(a.to_numpy(dtype=float), b.to_numpy(dtype=float), equal_nan=True)

req = """
SELECT pclass, fare, age, survived
FROM titanic
WHERE sex = 'male' AND age >= 18
ORDER BY fare DESC, age
LIMIT 10
"""
df_slice = titanic.loc[male_mask & age_mask, ['pclass', 'fare', 'age', 'survived']]
resultat_pandas = df_slice.sort_values(['fare', 'age'], ascending=[False, True]).head(10)

identique(sql(req), resultat_pandas)

True

## Exercices : corrections des 6 problèmes

Essaie d'abord de résoudre chaque problème toi-même, puis compare avec la requête SQL et le code pandas.

### Problème 1

In [82]:
sql("""
SELECT age, sex, survived
FROM titanic
WHERE age < 16 AND parch = 0
ORDER BY age
""")

,age,sex,survived
0,5.0,female,1
1,11.0,male,0
2,12.0,male,1
3,13.0,female,1
...,...,...,...
7,14.5,female,0
8,15.0,female,1
9,15.0,female,1
10,15.0,female,1


In [83]:
children = titanic.age < 16
parents = titanic.parch == 0
col_list = ['age', 'sex', 'survived']

titanic.loc[children & parents, col_list].sort_values('age')

,age,sex,survived
777,5.0,female,1
731,11.0,male,0
125,12.0,male,1
780,13.0,female,1
...,...,...,...
111,14.5,female,0
22,15.0,female,1
830,15.0,female,1
875,15.0,female,1


### Problème 2

In [84]:
sql("""
SELECT pclass, age, fare
FROM titanic
WHERE sex = 'female' AND pclass != 1
ORDER BY age DESC, fare DESC
LIMIT 20
""")

,pclass,age,fare
0,3,63.0,9.5875
1,2,57.0,10.5000
2,2,55.0,16.0000
3,2,54.0,23.0000
...,...,...,...
16,3,43.0,46.9000
17,2,42.0,26.0000
18,2,42.0,13.0000
19,3,41.0,39.6875


In [85]:
females = titanic.sex == 'female'
lower_class = titanic.pclass != 1
col_list = ['pclass', 'age', 'fare']

df_slice = titanic.loc[females & lower_class, col_list]
df_slice.nlargest(20, ['age', 'fare'])

,pclass,age,fare
483,3,63.0,9.5875
772,2,57.0,10.5000
15,2,55.0,16.0000
774,2,54.0,23.0000
...,...,...,...
678,3,43.0,46.9000
432,2,42.0,26.0000
865,2,42.0,13.0000
638,3,41.0,39.6875


### Problème 3

In [86]:
sql("""
SELECT *
FROM titanic
WHERE age < 18 AND pclass = 3 AND survived = 1
ORDER BY age, fare DESC
""")

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,1,3,male,0.42,0,1,8.5167,C,Third,child,0,None,Cherbourg,yes,0
1,1,3,female,0.75,2,1,19.2583,C,Third,child,0,None,Cherbourg,yes,0
2,1,3,female,0.75,2,1,19.2583,C,Third,child,0,None,Cherbourg,yes,0
3,1,3,male,1.00,1,2,20.5750,S,Third,child,0,None,Southampton,yes,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25,1,3,male,16.00,0,0,8.0500,S,Third,man,1,None,Southampton,yes,1
26,1,3,female,16.00,0,0,7.7500,Q,Third,woman,0,None,Queenstown,yes,1
27,1,3,female,16.00,0,0,7.7333,Q,Third,woman,0,None,Queenstown,yes,1
28,1,3,female,17.00,4,2,7.9250,S,Third,woman,0,None,Southampton,yes,0


In [87]:
children = titanic.age < 18
low_class = titanic.pclass == 3
survival = titanic.survived == 1

df_slice = titanic.loc[children & low_class & survival, :]
df_slice.sort_values(['age', 'fare'], ascending=[True, False])

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
803,1,3,male,0.42,0,1,8.5167,C,Third,child,False,NaN,Cherbourg,yes,False
469,1,3,female,0.75,2,1,19.2583,C,Third,child,False,NaN,Cherbourg,yes,False
644,1,3,female,0.75,2,1,19.2583,C,Third,child,False,NaN,Cherbourg,yes,False
788,1,3,male,1.00,1,2,20.5750,S,Third,child,False,NaN,Southampton,yes,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,1,3,male,16.00,0,0,8.0500,S,Third,man,True,NaN,Southampton,yes,True
208,1,3,female,16.00,0,0,7.7500,Q,Third,woman,False,NaN,Queenstown,yes,True
156,1,3,female,16.00,0,0,7.7333,Q,Third,woman,False,NaN,Queenstown,yes,True
68,1,3,female,17.00,4,2,7.9250,S,Third,woman,False,NaN,Southampton,yes,False


### Problème 4

In [88]:
sql("""
SELECT embarked, pclass, COUNT(*), AVG(fare)
FROM titanic
GROUP BY embarked, pclass
""")

,embarked,pclass,COUNT(*),AVG(fare)
0,None,1,2,80.000000
1,C,1,85,104.718529
2,C,2,17,25.358335
3,C,3,66,11.214083
...,...,...,...,...
6,Q,3,72,11.183393
7,S,1,127,70.364862
8,S,2,164,20.327439
9,S,3,353,14.644083


In [89]:
titanic.groupby(['embarked', 'pclass'], dropna=False).agg(
    passagers=('fare', 'size'),
    prix_moyen=('fare', 'mean'),
)

passagers  prix_moyen
embarked pclass                       
C        1              85  104.718529
         2              17   25.358335
         3              66   11.214083
Q        1               2   90.000000
...                    ...         ...
S        1             127   70.364862
         2             164   20.327439
         3             353   14.644083
NaN      1               2   80.000000

[10 rows x 2 columns]

### Problème 5

In [90]:
sql("""
SELECT pclass, survived, MIN(age), MAX(age), AVG(age)
FROM titanic
GROUP BY pclass, survived
""")

,pclass,survived,MIN(age),MAX(age),AVG(age)
0,1,0,2.00,71.0,43.695312
1,1,1,0.92,80.0,35.368197
2,2,0,16.00,70.0,33.544444
3,2,1,0.67,62.0,25.901566
4,3,0,1.00,74.0,26.555556
5,3,1,0.42,63.0,20.646118


In [91]:
titanic.groupby(['pclass', 'survived']).age.agg(['min', 'max', 'mean'])

min   max       mean
pclass survived                        
1      0          2.00  71.0  43.695312
       1          0.92  80.0  35.368197
2      0         16.00  70.0  33.544444
       1          0.67  62.0  25.901566
3      0          1.00  74.0  26.555556
       1          0.42  63.0  20.646118

### Problème 6

In [92]:
sql("""
SELECT survived, AVG(fare), MIN(age), MAX(age)
FROM titanic
GROUP BY survived
""")

,survived,AVG(fare),MIN(age),MAX(age)
0,0,22.117887,1.00,74.0
1,1,48.395408,0.42,80.0


In [93]:
titanic.groupby('survived').agg(
    prix_moyen=('fare', 'mean'),
    age_min=('age', 'min'),
    age_max=('age', 'max'),
)

,prix_moyen,age_min,age_max
survived,,,
0,22.117887,1.00,74.0
1,48.395408,0.42,80.0


### Contrôle automatique des 6 corrections

In [94]:
def meme_resultat(requete, resultat):
    a = sql(requete)
    b = resultat.copy()
    b.columns = a.columns
    # SQL renvoie None là où pandas met NaN : on harmonise avant de comparer
    a = a.where(a.notna(), np.nan)
    # à égalité sur la clé de tri, l'ordre des lignes peut différer : on compare les lignes triées
    a = a.sort_values(list(a.columns)).reset_index(drop=True)
    b = b.sort_values(list(b.columns)).reset_index(drop=True)
    pd.testing.assert_frame_equal(a, b, check_dtype=False)
    return True

print(meme_resultat(
    "SELECT age, sex, survived FROM titanic WHERE age < 16 AND parch = 0 ORDER BY age",
    titanic.loc[(titanic.age < 16) & (titanic.parch == 0), ['age', 'sex', 'survived']].sort_values('age')))

print(meme_resultat(
    "SELECT pclass, age, fare FROM titanic WHERE sex = 'female' AND pclass != 1 ORDER BY age DESC, fare DESC LIMIT 20",
    titanic.loc[(titanic.sex == 'female') & (titanic.pclass != 1), ['pclass', 'age', 'fare']].nlargest(20, ['age', 'fare'])))

print(meme_resultat(
    "SELECT survived, pclass, age, fare FROM titanic WHERE age < 18 AND pclass = 3 AND survived = 1 ORDER BY age, fare DESC",
    titanic.loc[(titanic.age < 18) & (titanic.pclass == 3) & (titanic.survived == 1), ['survived', 'pclass', 'age', 'fare']].sort_values(['age', 'fare'], ascending=[True, False])))

print(meme_resultat(
    "SELECT embarked, pclass, COUNT(*), AVG(fare) FROM titanic GROUP BY embarked, pclass",
    titanic.groupby(['embarked', 'pclass'], dropna=False).agg(passagers=('fare', 'size'), prix_moyen=('fare', 'mean')).reset_index()))

print(meme_resultat(
    "SELECT pclass, survived, MIN(age), MAX(age), AVG(age) FROM titanic GROUP BY pclass, survived",
    titanic.groupby(['pclass', 'survived']).age.agg(['min', 'max', 'mean']).reset_index()))

print(meme_resultat(
    "SELECT survived, AVG(fare), MIN(age), MAX(age) FROM titanic GROUP BY survived",
    titanic.groupby('survived').agg(prix_moyen=('fare', 'mean'), age_min=('age', 'min'), age_max=('age', 'max')).reset_index()))

True
True
True
True
True
True
